# DSDE Election OCR Pipeline
**Flow:** PDF → typhoon-ocr (image→text) → save .txt → typhoon-v2.5-30b (text→JSON) → save JSON+CSV

**Rate limits:**
- `typhoon-ocr`: 2 req/s, 20 req/min
- `typhoon-v2.5-30b-a3b-instruct`: 5 req/s, 200 req/min

**Run order:** Cell 1 → 2 → 3 → 4 → 5 → 6

In [ ]:
# ============================================================
# CELL 1: Install + Imports
# ============================================================
!pip install typhoon-ocr pdf2image Pillow python-dotenv requests -q
!apt-get install -y poppler-utils -q

import os, re, json, csv, shutil, time, tempfile, random
from pathlib import Path
from pdf2image import convert_from_path
from dotenv import load_dotenv
import requests as req

load_dotenv()
print("Done")

In [ ]:
# ============================================================
# CELL 2: Config — EDIT THIS CELL
# ============================================================
TYPHOON_KEY    = os.getenv("TYPHOON_KEY", "")   # from .env

PROVINCE       = "อุบลราชธานี"
CONSTITUENCY   = 2
RAW_DATA_DIR   = "rawData"
OUTPUT_DIR     = "output"
OCR_TEXT_DIR   = "ocr_texts"    # raw OCR text saved here per file

Path(OUTPUT_DIR).mkdir(exist_ok=True)
Path(OCR_TEXT_DIR).mkdir(exist_ok=True)

# API endpoints
OCR_BASE_URL   = "https://api.opentyphoon.ai/v1"
LLM_BASE_URL   = "https://api.opentyphoon.ai/v1"
OCR_MODEL      = "typhoon-ocr"
LLM_MODEL      = "typhoon-v2.5-30b-a3b-instruct"

# Rate limit settings (from image)
OCR_DELAY      = 0.5   # seconds between OCR calls (safe for 2/s, 20/min)
LLM_DELAY      = 0.5   # seconds between LLM calls (safe for 5/s, 200/min)
OCR_PAGE_TIMEOUT  = 60    # seconds per page before skipping
FILE_TIMEOUT      = 180 

# Set API key for typhoon_ocr library
os.environ["TYPHOON_OCR_API_KEY"] = TYPHOON_KEY
os.environ["OPENAI_API_KEY"]      = TYPHOON_KEY

print(f"TYPHOON_KEY   : {'set' if TYPHOON_KEY else 'MISSING'}")
print(f"RAW_DATA_DIR  : {RAW_DATA_DIR}")
print(f"OCR texts     : {OCR_TEXT_DIR}/")
print(f"Output        : {OUTPUT_DIR}/")
print(f"OCR delay     : {OCR_DELAY}s (max 20 req/min)")
print(f"LLM delay     : {LLM_DELAY}s (max 200 req/min)")

In [ ]:
# ============================================================
# CELL 3: Helper functions
# ============================================================

def thai_to_int(s: str) -> int:
    thai = "๐๑๒๓๔๕๖๗๘๙"
    return int("".join(str(thai.index(c)) if c in thai else c
                       for c in str(s).strip()))

def pdf_to_images(pdf_path: str, dpi: int = 300) -> list:
    """Convert PDF → images, handles Thai filenames on Windows."""
    tmp = None
    try:
        with tempfile.NamedTemporaryFile(suffix=".pdf", delete=False) as f:
            tmp = f.name
        shutil.copy2(pdf_path, tmp)
        return convert_from_path(tmp, dpi=dpi)
    finally:
        if tmp and os.path.exists(tmp):
            os.unlink(tmp)

def save_image_tmp(page_img, index: int) -> str:
    """Save PIL image to temp PNG, return path."""
    tmp_dir = Path("_tmp_imgs")
    tmp_dir.mkdir(exist_ok=True)
    path = tmp_dir / f"page_{index:03d}.png"
    page_img.save(str(path), "PNG")
    return str(path)

def llm_json_path(pdf_path: str) -> Path:
    """
    Mirror the input folder structure in output dir.
    e.g. rawData/อำเภอ/ตำบล/หน่วยที่ 1/5ทับ18.pdf
      →  output/อำเภอ/ตำบล/หน่วยที่ 1/5ทับ18.json
    """
    pdf     = Path(pdf_path)
    raw_dir = Path(RAW_DATA_DIR)

    # Get relative path from rawData root
    try:
        rel = pdf.relative_to(raw_dir)
    except ValueError:
        rel = Path(pdf.name)

    # Build mirrored output path
    out = Path(OUTPUT_DIR) / rel.with_suffix(".json")
    out.parent.mkdir(parents=True, exist_ok=True)   # create subfolders
    return out

def ocr_txt_path(pdf_path: str) -> Path:
    """Mirror input structure in ocr_texts dir."""
    pdf     = Path(pdf_path)
    raw_dir = Path(RAW_DATA_DIR)
    try:
        rel = pdf.relative_to(raw_dir)
    except ValueError:
        rel = Path(pdf.name)

    # ← change: use stem + manual suffix instead of with_suffix
    out = Path(OCR_TEXT_DIR) / rel.parent / (pdf.stem + "_ocr.txt")
    out.parent.mkdir(parents=True, exist_ok=True)
    return out

print("Helpers loaded")

In [ ]:
# ============================================================
# CELL 4: OCR function (typhoon-ocr) + LLM parse function
# ============================================================
from typhoon_ocr import ocr_document

VALID_PARTIES = """
ประชาธิปัตย์, ประชากรไทย, ความหวังใหม่, เพื่อไทย, ภูมิใจไทย,
สังคมประชาธิปไตยไทย, รักชาติ, ประชาธิปไตยใหม่, ครูไทยเพื่อประชาชน,
ประชาชน, ไทยก้าวใหม่, เสรีรวมไทย, พลังไทยรักชาติ, เพื่อชีวิตใหม่,
ทางเลือกใหม่, เศรษฐกิจ, สร้างอนาคตไทย, พลังธรรมใหม่, ไทยธรรม,
ไทยพร้อม, ปวงชนไทย, เพื่อชาติไทย, ประชาชาติ, แผ่นดินธรรม, คลองไทย,
พลังประชารัฐ, เป็นธรรม, พลังเพื่อไทย, ประชาไทย, กรีน, วิชชั่นใหม่,
พลวัต, กล้าธรรม, ไทยรวมไทย, ฟิวชัน, พลังสังคมใหม่, ไทยสร้างไทย,
รวมไทยสร้างชาติ, มิติใหม่, ไทยภักดี, ไทยพิทักษ์ธรรม, ไทยชนะ,
ไทรวมพลัง, โอกาสใหม่, ท้องที่ไทย, ใหม่, แรงงานสร้างชาติ, ไทยก้าวหน้า,
พร้อม, รวมใจไทย, ประชาอาสาชาติ, ไทยทรัพย์ทวี, รวมพลังประชาชน,
เพื่อบ้านเมือง, รวมพลัง, อนาคตไทย, เครือข่ายชาวนาแห่งประเทศไทย
"""

def ocr_pdf_to_text(pdf_path: str) -> str:
    """
    Step 1: PDF → OCR text via typhoon-ocr API
    Each page has OCR_PAGE_TIMEOUT seconds before being skipped.
    """
    txt_path = ocr_txt_path(pdf_path)

    if txt_path.exists() and txt_path.stat().st_size > 50:
        print(f"    ↩️  OCR cached: {txt_path.name}")
        return txt_path.read_text(encoding="utf-8")

    pages    = pdf_to_images(pdf_path)
    all_text = []

    for i, page in enumerate(pages):
        img_path = save_image_tmp(page, i)
        print(f"    🔍 OCR page {i+1}/{len(pages)}...", end=" ", flush=True)

        # ── Wrap ocr_document in thread with real timeout ─
        page_result = {}
        page_error  = {}

        def _ocr(_img=img_path):
            try:
                # No timeout= param — typhoon_ocr doesn't support it
                md = ocr_document(
                    _img,
                    model     = OCR_MODEL,
                    task_type = "v1.5",
                )
                page_result["text"] = md
            except Exception as e:
                page_error["err"] = str(e)

        pt = threading.Thread(target=_ocr)
        pt.start()
        pt.join(timeout=OCR_PAGE_TIMEOUT)   # ← real timeout via thread

        if pt.is_alive():
            # Still running after timeout → skip this page
            print(f"⏱️ page timeout ({OCR_PAGE_TIMEOUT}s) → skipped")
            all_text.append("")
            # Note: daemon thread will eventually finish on its own
            continue

        if "err" in page_error:
            err = page_error["err"]
            if "429" in err or "rate" in err.lower():
                print(f"⚠️ Rate limit → wait 60s")
                time.sleep(60)
                all_text.append("")
            else:
                print(f"❌ {err[:80]}")
                all_text.append("")
            continue

        md = page_result.get("text", "")
        all_text.append(md)
        print(f"✓ ({len(md)} chars)")
        time.sleep(OCR_DELAY)   # rate limit delay after success

    full_text = "\n\n--- PAGE BREAK ---\n\n".join(all_text)

    if full_text.strip():
        txt_path.write_text(full_text, encoding="utf-8")
        print(f"    💾 Saved → {txt_path.name}")
    else:
        print(f"    ⚠️ All pages empty — not cached")

    return full_text

def llm_parse_to_json(ocr_text: str, metadata: dict) -> dict:
    """
    Step 2: OCR text → structured JSON via typhoon-v2.5-30b
    Rate limit: 5 req/s, 200 req/min → wait LLM_DELAY between calls
    """
    # ── Use metadata to determine form type reliably ──────
    
    # Check metadata first (most reliable)
    is_party = "บช" in metadata.get("source", "")

    # Update prompt to also read from the bracket in the image
    prompt = f"""You are a Thai election data extractor.
    Extract all data from this OCR text of a Thai election form and return ONLY valid JSON.

    IMPORTANT - Determine form type by reading the bracket text at the top of the form:
    - If you see "(แบบบัญชีรายชื่อ)" → form_type = "party_list"
    - If you see "(แบบแบ่งเขตเลือกตั้ง)" → form_type = "constituency"
    - If unclear, use this hint from filename: {"party_list" if is_party else "constituency"}

    Valid party names (fix misspellings):
    {VALID_PARTIES}

    Return this JSON structure:
    {json.dumps({
        "form_type": "party_list OR constituency — read from bracket",
        "summary": {
            "eligible_voters":   0,
            "turnout":           0,
            "ballots_allocated": 0,
            "ballots_used":      0,
            "valid_ballots":     0,
            "spoiled_ballots":   0,
            "abstain_ballots":   0,
            "ballots_remaining": 0,
        },
        "results": [
            {
                "number": 0,
                "party": "<party name corrected>",
                "votes": 0,
                "votes_th": "<thai word for votes from bracket e.g. สี่ร้อยหกสิบแปด>"
            }
        ] 
    }, ensure_ascii=False)}

    Rules:
    - Convert Thai digits to Arabic integers
    - Fix party name OCR typos against the valid list
    - party_th / name_th = original Thai text exactly as written
    - Return ONLY JSON, no explanation

    OCR TEXT:
    {ocr_text[:6000]}"""

    headers = {
        "Authorization": f"Bearer {TYPHOON_KEY}",
        "Content-Type":  "application/json",
    }
    body = {
        "model":       LLM_MODEL,
        "messages":    [{"role": "user", "content": prompt}],
        "max_tokens":  2048,
        "temperature": 0.1,
    }

    for attempt in range(3):
        try:
            time.sleep(LLM_DELAY)
            resp = req.post(
                f"{LLM_BASE_URL}/chat/completions",
                headers = headers,
                json    = body,
                timeout = 60
            )

            if resp.status_code == 429:
                wait = 30 * (attempt + 1)
                print(f"    ⚠️ Rate limit → wait {wait}s")
                time.sleep(wait)
                continue

            resp.raise_for_status()
            raw = resp.json()["choices"][0]["message"]["content"].strip()
            raw = re.sub(r'^```json\s*', '', raw)
            raw = re.sub(r'\s*```$',     '', raw).strip()
            data = json.loads(raw)

            llm_form_type  = data.get("form_type", "")
            meta_form_type = "party_list" if is_party else "constituency"
            if llm_form_type in ("party_list", "constituency"):
                final_form_type = llm_form_type
            else:
                final_form_type = meta_form_type
                print(f"    ⚠️ form_type invalid ('{llm_form_type}') → using: {meta_form_type}")
            data["form_type"] = final_form_type

            results  = data.get("results", [])
            vote_sum = sum(r.get("votes", 0) for r in results)
            return {
                "metadata":    metadata,
                "form_type":   data["form_type"],
                "summary":     data.get("summary", {}),
                "results":     sorted(results, key=lambda x: x.get("number", 0)),
                "_validation": {
                    "total_votes_in_table": vote_sum,
                    "status": "PASS" if vote_sum > 0 else "EMPTY",
                }
            }

        except json.JSONDecodeError as e:
            print(f"    ⚠️ JSON parse error attempt {attempt+1}: {e}")
            print(f"    Raw: {raw[:200]}")
        except Exception as e:
            print(f"    ❌ LLM error attempt {attempt+1}: {e}")
            time.sleep(5)

    return {
        "metadata":    metadata,
        "form_type":   "party_list" if is_party else "constituency",
        "summary":     {},
        "results":     [],
        "_validation": {"total_votes_in_table": 0, "status": "FAILED"}
    }
print("OCR + LLM functions loaded")

In [ ]:
# ============================================================
# CELL 5: Test on ONE file
# Run this before batch to verify pipeline works
# ============================================================
TEST_PDF = r"rawData\เขต 2 อ.เมืองฯ\เขตไร่น้อย\เขตไร่น้อย\หน่วยเลือกตั้งที่ 1\5ทับ18.pdf"

print(f"Testing: {TEST_PDF}")
print(f"Exists : {Path(TEST_PDF).exists()}")
print()

# Step 1: OCR
print("Step 1: OCR → text")
ocr_text = ocr_pdf_to_text(TEST_PDF)
print(f"OCR text ({len(ocr_text)} chars):")
print(ocr_text[:800])
print()

# Step 2: LLM parse
print("Step 2: LLM → JSON")
meta = {
    "province": PROVINCE, "constituency": CONSTITUENCY,
    "source": TEST_PDF, "ocr_engine": OCR_MODEL, "llm_parser": LLM_MODEL
}
result = llm_parse_to_json(ocr_text, meta)

print(json.dumps(result, ensure_ascii=False, indent=2))
print()
print(f"Validation: {result['_validation']['status']}")
print(f"Results   : {len(result.get('results', []))} rows")

# Save test JSON
test_path = Path(OUTPUT_DIR) / "test_single.json"
test_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\nSaved → {test_path}")

In [ ]:
# ============================================================
# CELL 6: Batch ALL files
# Rate limits:
#   typhoon-ocr:  20 req/min → OCR_DELAY=3.5s
#   typhoon-30b: 200 req/min → LLM_DELAY=0.5s
# For 566 PDFs × avg 2 pages = ~1132 OCR calls
#   At 20/min → ~57 min OCR | At 200/min → ~6 min LLM
# ============================================================
import threading

all_pdfs    = sorted(Path(RAW_DATA_DIR).rglob("*.pdf"))
all_records = []
failed      = []
ocr_calls   = 0
llm_calls   = 0

print(f"Found      : {len(all_pdfs)} PDFs")
print(f"OCR rate   : 20 req/min (delay={OCR_DELAY}s)")
print(f"LLM rate   : 200 req/min (delay={LLM_DELAY}s)")
print(f"Est. time  : ~{len(all_pdfs)*2//20 + len(all_pdfs)//200 + 10} min")
print(f"Output dir : {Path(OUTPUT_DIR).resolve()}")
print()

for i, pdf in enumerate(all_pdfs):

    # ── Extract metadata from folder structure ────────────
    parts = pdf.parts
    try:
        amphoe   = parts[-4]
        tambon   = parts[-3]
        m        = re.search(r"(\d+)$", parts[-2])
        unit_num = int(m.group(1)) if m else 0
    except:
        amphoe, tambon, unit_num = "unknown", "unknown", 0

    is_party   = "บช" in pdf.name
    form_label = "party_list" if is_party else "constituency"

    # ── Skip if already done ──────────────────────────────
    out_json = llm_json_path(str(pdf))
    if out_json.exists() and out_json.stat().st_size > 50:
        print(f"[{i+1}/{len(all_pdfs)}] SKIP: {pdf.name}")
        try:
            done = json.loads(out_json.read_text(encoding="utf-8"))
            for r in done.get("results", []):
                all_records.append({
                    "province":          PROVINCE,
                    "constituency":      CONSTITUENCY,
                    "amphoe":            amphoe,
                    "tambon":            tambon,
                    "unit":              unit_num,
                    "form_type":         form_label,
                    "eligible_voters":   done["summary"].get("eligible_voters"),
                    "turnout":           done["summary"].get("turnout"),
                    "ballots_allocated": done["summary"].get("ballots_allocated"),
                    "ballots_used":      done["summary"].get("ballots_used"),
                    "valid_ballots":     done["summary"].get("valid_ballots"),
                    "spoiled_ballots":   done["summary"].get("spoiled_ballots"),
                    "abstain_ballots":   done["summary"].get("abstain_ballots"),
                    "ballots_remaining": done["summary"].get("ballots_remaining"),
                    "candidate_number":  r.get("number"),
                    "candidate_name":    r.get("name", ""),
                    "party":             r.get("party", ""),
                    "votes":             r.get("votes", 0),
                    "votes_th":          r.get("votes_th", ""),
                    "validation":        done.get("_validation", {}).get("status", ""),
                    "source":            str(pdf),
                })
        except: pass
        continue

    print(f"[{i+1}/{len(all_pdfs)}] {amphoe}/{tambon}/หน่วย{unit_num} — {pdf.name}")

    # ── Process with 3 min timeout ────────────────────────
    result_container = {}
    error_container  = {}

    def process_file(
        _pdf=pdf, _amphoe=amphoe, _tambon=tambon,
        _unit_num=unit_num, _form_label=form_label
    ):
        try:
            ocr_text = ocr_pdf_to_text(str(_pdf))
            if not ocr_text.strip():
                raise ValueError("Empty OCR output")
            meta = {
                "province":     PROVINCE,
                "constituency": CONSTITUENCY,
                "amphoe":       _amphoe,
                "tambon":       _tambon,
                "unit":         _unit_num,
                "form_type":    _form_label,
                "source":       str(_pdf),
                "ocr_engine":   OCR_MODEL,
                "llm_parser":   LLM_MODEL,
            }
            result_container["result"] = llm_parse_to_json(ocr_text, meta)
        except Exception as e:
            error_container["error"] = str(e)

    t = threading.Thread(target=process_file)
    t.start()
    t.join(timeout=180)   # 3 min max per file

    if t.is_alive():
        print(f"  ⏱️ TIMEOUT after 3 min — skipping")
        failed.append({"file": str(pdf), "error": "timeout after 3 min"})
        continue

    if "error" in error_container:
        print(f"  ✗ FAILED: {error_container['error']}")
        failed.append({"file": str(pdf), "error": error_container["error"]})
        continue

    result = result_container.get("result")
    if not result:
        print(f"  ✗ No result returned")
        failed.append({"file": str(pdf), "error": "no result"})
        continue

    # ── Save JSON ─────────────────────────────────────────
    out_json.write_text(
        json.dumps(result, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )
    ocr_calls += 1
    llm_calls += 1

    # ── Collect CSV rows ──────────────────────────────────
    for r in result.get("results", []):
        all_records.append({
            "province":          PROVINCE,
            "constituency":      CONSTITUENCY,
            "amphoe":            amphoe,
            "tambon":            tambon,
            "unit":              unit_num,
            "form_type":         form_label,
            "eligible_voters":   result["summary"].get("eligible_voters"),
            "turnout":           result["summary"].get("turnout"),
            "ballots_allocated": result["summary"].get("ballots_allocated"),
            "ballots_used":      result["summary"].get("ballots_used"),
            "valid_ballots":     result["summary"].get("valid_ballots"),
            "spoiled_ballots":   result["summary"].get("spoiled_ballots"),
            "abstain_ballots":   result["summary"].get("abstain_ballots"),
            "ballots_remaining": result["summary"].get("ballots_remaining"),
            "candidate_number":  r.get("number"),
            "candidate_name":    r.get("name", ""),
            "party":             r.get("party", ""),
            "votes":             r.get("votes", 0),
            "votes_th":          r.get("votes_th", ""),
            "validation":        result["_validation"]["status"],
            "source":            str(pdf),
        })

    v = result["_validation"]
    print(f"  ✓ {v['status']} rows={len(result['results'])} total_votes={v['total_votes_in_table']}")
    print(f"  calls: OCR={ocr_calls} LLM={llm_calls}")

# ── Save master CSV ───────────────────────────────────────
if all_records:
    csv_path = Path(OUTPUT_DIR) / "all_results.csv"
    with open(csv_path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=list(all_records[0].keys()))
        writer.writeheader()
        writer.writerows(all_records)
    print(f"\nMaster CSV → {csv_path} ({len(all_records)} rows)")

# ── Save failure log ──────────────────────────────────────
if failed:
    fail_path = Path(OUTPUT_DIR) / "_failed.json"
    fail_path.write_text(json.dumps(failed, ensure_ascii=False, indent=2))
    print(f"Failed: {len(failed)} → {fail_path}")

ok = len(all_pdfs) - len(failed)
print(f"\nDone! ✓={ok} ✗={len(failed)} Total={len(all_pdfs)}")
print(f"Total OCR calls : {ocr_calls}")
print(f"Total LLM calls : {llm_calls}")
print(f"Results saved   : {Path(OUTPUT_DIR).resolve()}")

In [ ]:
# ============================================================
# CELL 7: Retry failed files
# ============================================================
fail_path = Path(OUTPUT_DIR) / "_failed.json"
if not fail_path.exists():
    print("No failures to retry!")
else:
    failed_list = json.loads(fail_path.read_text())
    print(f"Retrying {len(failed_list)} files...")
    still_failed = []

    for item in failed_list:
        pdf  = Path(item["file"])
        parts = pdf.parts
        try:
            amphoe   = parts[-4]
            tambon   = parts[-3]
            m        = re.search(r"(\d+)$", parts[-2])
            unit_num = int(m.group(1)) if m else 0
        except:
            amphoe, tambon, unit_num = "unknown", "unknown", 0

        is_party   = "บช" in pdf.name
        form_label = "party_list" if is_party else "constituency"
        print(f"Retrying {pdf.name}...", end=" ")

        try:
            # Delete cached OCR if it was the OCR that failed
            txt = ocr_txt_path(str(pdf))
            if txt.exists() and txt.stat().st_size < 100:
                txt.unlink()  # delete empty/bad cache

            ocr_text = ocr_pdf_to_text(str(pdf))
            meta     = {"province": PROVINCE, "constituency": CONSTITUENCY,
                        "amphoe": amphoe, "tambon": tambon, "unit": unit_num,
                        "source": str(pdf)}
            result   = llm_parse_to_json(ocr_text, meta)
            llm_json_path(str(pdf)).write_text(
                json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8"
            )
            print(f"✓ {result['_validation']['status']}")
        except Exception as e:
            print(f"✗ {e}")
            still_failed.append(item)

    if still_failed:
        fail_path.write_text(json.dumps(still_failed, ensure_ascii=False, indent=2))
        print(f"\n{len(still_failed)} still failing")
    else:
        fail_path.unlink()
        print("\nAll retried successfully!")